# Sentiment Analysis of Political News — Dissertation Pipeline
**Models:** VADER (baseline) · BERT · RoBERTa · DistilBERT  
**Dataset:** All the News (Kaggle) — filtered to political articles, auto-labelled via VADER  
**Evaluation:** Accuracy · Precision · Recall · F1-score (macro)

---
## ⚙️ SETUP: Run this cell first — installs everything

In [ ]:
# Install all required libraries
!pip install transformers datasets torch vaderSentiment scikit-learn pandas numpy matplotlib seaborn kaggle --quiet

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    RobertaTokenizer, RobertaForSequenceClassification,
    DistilBertTokenizer, DistilBertForSequenceClassification,
    Trainer, TrainingArguments
)
from datasets import Dataset
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 📦 STEP 1 — Load Dataset

**Option A (recommended):** Upload `articles1.csv` from Kaggle "All the News" dataset  
Download from: https://www.kaggle.com/datasets/snapcrack/all-the-news  
Then upload to Colab using the file upload button.

**Option B (no download needed):** Uses a smaller built-in demo dataset to test the pipeline first.

In [ ]:
# ============================================================
# OPTION A — Use this if you uploaded articles1.csv from Kaggle
# ============================================================
USE_KAGGLE_DATA = False  # Set to True once you upload articles1.csv

if USE_KAGGLE_DATA:
    df_raw = pd.read_csv('articles1.csv')
    print(f'Raw dataset shape: {df_raw.shape}')
    print(df_raw.columns.tolist())

    # Filter to political news only
    political_keywords = [
        'election', 'president', 'senate', 'congress', 'democrat', 'republican',
        'government', 'policy', 'trump', 'clinton', 'obama', 'political', 'vote',
        'parliament', 'minister', 'legislation', 'campaign', 'white house'
    ]
    pattern = '|'.join(political_keywords)
    df = df_raw[
        df_raw['title'].str.lower().str.contains(pattern, na=False) |
        df_raw['content'].str.lower().str.contains(pattern, na=False)
    ].copy()

    # Use title + first 300 chars of content as the text input
    df['text'] = df['title'].fillna('') + '. ' + df['content'].fillna('').str[:300]
    df = df[['text']].dropna().reset_index(drop=True)

    # Sample 8000 for manageable training time
    if len(df) > 8000:
        df = df.sample(8000, random_state=42).reset_index(drop=True)

    print(f'Political articles after filtering: {len(df)}')

else:
    # ============================================================
    # OPTION B — Small demo dataset (tests the full pipeline)
    # Replace this with Option A for your actual dissertation run
    # ============================================================
    demo_texts = [
        "The president signed a landmark healthcare bill today, praising it as a major victory for citizens.",
        "Opposition leaders condemned the new tax policy as deeply unfair and economically disastrous.",
        "The Senate committee met to review the proposed budget changes without reaching a conclusion.",
        "Voters expressed strong enthusiasm for the candidate's economic reform proposals.",
        "The government's handling of the crisis has been widely criticised as incompetent and dangerous.",
        "Parliament passed the education reform bill with bipartisan support from both parties.",
        "The election results remain uncertain as counting continues in several key districts.",
        "Protesters gathered outside the White House demanding action on climate legislation.",
        "The minister announced a new infrastructure investment plan worth billions of dollars.",
        "Critics argue the immigration policy violates fundamental human rights principles.",
        "The prime minister delivered an optimistic speech about the country's economic recovery.",
        "Allegations of corruption have damaged the ruling party's approval ratings significantly.",
        "Congressional leaders agreed to extend the debt ceiling deadline by three months.",
        "The foreign policy decision has drawn sharp condemnation from international allies.",
        "Community leaders praised the local government's response to the housing crisis.",
    ] * 40  # Repeat to get 600 samples for demo

    df = pd.DataFrame({'text': demo_texts})
    print(f'Demo dataset size: {len(df)} samples')
    print('NOTE: Switch USE_KAGGLE_DATA=True and upload articles1.csv for your real dissertation run')

---
## 🏷️ STEP 2 — Auto-Label Using VADER
This replicates the methodology from Samin et al. (2024) — Paper 6 in your literature review.

In [ ]:
# VADER auto-labelling (justified by Samin et al., 2024)
analyzer = SentimentIntensityAnalyzer()

def vader_label(text):
    score = analyzer.polarity_scores(str(text))['compound']
    if score > 0.05:
        return 2  # Positive
    elif score < -0.05:
        return 0  # Negative
    else:
        return 1  # Neutral

df['label'] = df['text'].apply(vader_label)

# Label distribution
label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
dist = df['label'].value_counts().sort_index()
print('Label distribution:')
for k, v in dist.items():
    print(f'  {label_map[k]}: {v} ({v/len(df)*100:.1f}%)')

# Plot
plt.figure(figsize=(6,4))
sns.barplot(x=[label_map[i] for i in dist.index], y=dist.values, palette=['#e74c3c','#95a5a6','#2ecc71'])
plt.title('Dataset Label Distribution (VADER Auto-Labelling)')
plt.ylabel('Count')
plt.xlabel('Sentiment')
plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150)
plt.show()
print('Saved: label_distribution.png')

---
## ✂️ STEP 3 — Train / Validation / Test Split

In [ ]:
# 70% train / 15% val / 15% test — standard split
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# Save test set for consistent evaluation across all models
test_df.to_csv('test_set.csv', index=False)
print('Test set saved — same test set used for all 4 models (fair comparison)')

---
## 📊 RUN 1 — VADER Baseline
No training. Evaluates VADER directly on the test set.  
**Note:** Since labels were generated by VADER, this tests consistency — in your dissertation, note this and use it as a methodological comparison point.

In [ ]:
# VADER evaluation on test set
test_df_vader = pd.read_csv('test_set.csv')
test_df_vader['vader_pred'] = test_df_vader['text'].apply(vader_label)

y_true = test_df_vader['label'].values
y_pred = test_df_vader['vader_pred'].values

acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

print('=' * 50)
print('RUN 1 — VADER BASELINE RESULTS')
print('=' * 50)
print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {p:.4f} (macro)')
print(f'Recall:    {r:.4f} (macro)')
print(f'F1-score:  {f1:.4f} (macro)')
print()
print(classification_report(y_true, y_pred, target_names=['Negative','Neutral','Positive']))

# Store results
results = []
results.append({'Model': 'VADER (Baseline)', 'Accuracy': acc, 'Precision': p, 'Recall': r, 'F1-Score': f1})

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Neg','Neu','Pos'], yticklabels=['Neg','Neu','Pos'])
plt.title('VADER — Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('cm_vader.png', dpi=150)
plt.show()

---
## 🤖 HELPER FUNCTION — Fine-Tune Any HuggingFace Model
Used for Runs 2, 3, and 4. Keeps code clean and consistent.

In [ ]:
def fine_tune_and_evaluate(model_name, tokenizer_class, model_class, run_label,
                            train_df, val_df, test_df,
                            max_length=256, epochs=3, batch_size=16, lr=2e-5):
    """
    Fine-tunes a HuggingFace sequence classification model and returns evaluation metrics.
    """
    print(f'\n{"="*55}')
    print(f'  {run_label} — Loading: {model_name}')
    print(f'{"="*55}')

    # Load tokenizer and model
    tokenizer = tokenizer_class.from_pretrained(model_name)
    model = model_class.from_pretrained(model_name, num_labels=3, ignore_mismatched_sizes=True)

    # Tokenise
    def tokenize(batch):
        return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=max_length)

    train_hf = Dataset.from_pandas(train_df[['text','label']].reset_index(drop=True))
    val_hf   = Dataset.from_pandas(val_df[['text','label']].reset_index(drop=True))
    test_hf  = Dataset.from_pandas(test_df[['text','label']].reset_index(drop=True))

    train_hf = train_hf.map(tokenize, batched=True)
    val_hf   = val_hf.map(tokenize, batched=True)
    test_hf  = test_hf.map(tokenize, batched=True)

    train_hf.set_format('torch', columns=['input_ids','attention_mask','label'])
    val_hf.set_format('torch', columns=['input_ids','attention_mask','label'])
    test_hf.set_format('torch', columns=['input_ids','attention_mask','label'])

    # Training arguments
    safe_name = run_label.replace(' ','_').replace('/','_')
    args = TrainingArguments(
        output_dir=f'./results_{safe_name}',
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        warmup_ratio=0.1,
        weight_decay=0.01,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to='none'
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        acc = accuracy_score(labels, preds)
        p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
        return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_hf,
        eval_dataset=val_hf,
        compute_metrics=compute_metrics
    )

    # Train
    print(f'Training for {epochs} epochs...')
    trainer.train()

    # Evaluate on test set
    print('Evaluating on test set...')
    preds_output = trainer.predict(test_hf)
    y_pred = np.argmax(preds_output.predictions, axis=-1)
    y_true = test_df['label'].values

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

    print(f'\n--- {run_label} TEST RESULTS ---')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {p:.4f}')
    print(f'Recall:    {r:.4f}')
    print(f'F1-score:  {f1:.4f}')
    print(classification_report(y_true, y_pred, target_names=['Negative','Neutral','Positive']))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Neg','Neu','Pos'], yticklabels=['Neg','Neu','Pos'])
    plt.title(f'{run_label} — Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'cm_{safe_name}.png', dpi=150)
    plt.show()

    return {'Model': run_label, 'Accuracy': acc, 'Precision': p, 'Recall': r, 'F1-Score': f1}

print('Helper function loaded. Ready to run models.')

---
## 🤖 RUN 2 — BERT Fine-Tuning
Expected time: ~30–40 minutes on Colab T4 GPU

In [ ]:
bert_results = fine_tune_and_evaluate(
    model_name='bert-base-uncased',
    tokenizer_class=BertTokenizer,
    model_class=BertForSequenceClassification,
    run_label='BERT',
    train_df=train_df,
    val_df=val_df,
    test_df=test_df
)
results.append(bert_results)

---
## 🤖 RUN 3 — RoBERTa Fine-Tuning
Expected time: ~35–45 minutes on Colab T4 GPU

In [ ]:
roberta_results = fine_tune_and_evaluate(
    model_name='roberta-base',
    tokenizer_class=RobertaTokenizer,
    model_class=RobertaForSequenceClassification,
    run_label='RoBERTa',
    train_df=train_df,
    val_df=val_df,
    test_df=test_df
)
results.append(roberta_results)

---
## 🤖 RUN 4 — DistilBERT Fine-Tuning
Expected time: ~20–25 minutes on Colab T4 GPU (fastest of the three)

In [ ]:
distilbert_results = fine_tune_and_evaluate(
    model_name='distilbert-base-uncased',
    tokenizer_class=DistilBertTokenizer,
    model_class=DistilBertForSequenceClassification,
    run_label='DistilBERT',
    train_df=train_df,
    val_df=val_df,
    test_df=test_df
)
results.append(distilbert_results)

---
## 📊 FINAL RESULTS — Comparison Table + Charts
This is what goes directly into your dissertation Results section.

In [ ]:
# Results table
results_df = pd.DataFrame(results)
results_df = results_df.round(4)
print('\n' + '='*65)
print('  FINAL COMPARISON — ALL MODELS')
print('='*65)
print(results_df.to_string(index=False))
print('='*65)

# Save to CSV
results_df.to_csv('final_results.csv', index=False)
print('\nSaved: final_results.csv')

In [ ]:
# Bar chart — all metrics side by side
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
models  = results_df['Model'].tolist()
x = np.arange(len(metrics))
width = 0.18
colors = ['#95a5a6', '#3498db', '#e67e22', '#2ecc71']

fig, ax = plt.subplots(figsize=(11, 6))
for i, (model, color) in enumerate(zip(models, colors)):
    vals = results_df[results_df['Model'] == model][metrics].values.flatten()
    bars = ax.bar(x + i * width, vals, width, label=model, color=color, alpha=0.88)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Sentiment Analysis of Political News')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.08)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison_chart.png', dpi=150)
plt.show()
print('Saved: model_comparison_chart.png')

In [ ]:
# F1-score ranked bar chart (cleaner single-metric view)
results_sorted = results_df.sort_values('F1-Score', ascending=True)
colors_sorted = ['#95a5a6' if m == 'VADER (Baseline)' else '#3498db' if m == 'BERT'
                 else '#e67e22' if m == 'RoBERTa' else '#2ecc71'
                 for m in results_sorted['Model']]

plt.figure(figsize=(7, 4))
bars = plt.barh(results_sorted['Model'], results_sorted['F1-Score'], color=colors_sorted, alpha=0.88)
for bar, val in zip(bars, results_sorted['F1-Score']):
    plt.text(val - 0.02, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', ha='right', color='white', fontweight='bold')
plt.xlabel('Macro F1-Score')
plt.title('Model Ranking by F1-Score — Political News Sentiment')
plt.xlim(0, 1.0)
plt.tight_layout()
plt.savefig('f1_ranking.png', dpi=150)
plt.show()
print('Saved: f1_ranking.png')

---
## 📋 STEP 5 — Download All Files
Run this to download everything you need for your dissertation.

In [ ]:
from google.colab import files
import os

# Files to download
output_files = [
    'final_results.csv',
    'label_distribution.png',
    'model_comparison_chart.png',
    'f1_ranking.png',
    'cm_vader.png',
    'cm_BERT.png',
    'cm_RoBERTa.png',
    'cm_DistilBERT.png',
]

for f in output_files:
    if os.path.exists(f):
        files.download(f)
        print(f'Downloaded: {f}')
    else:
        print(f'Not found (model may not have run yet): {f}')

---
## 📝 DISSERTATION WRITE-UP REFERENCE

### What to write in each section based on this notebook:

**Methodology:**
- Dataset: All the News (Kaggle, snapcrack) filtered to political articles using keyword matching; auto-labelled using VADER (compound score threshold: >0.05 positive, <-0.05 negative, else neutral) following Samin et al. (2024)
- Train/Val/Test: 70/15/15 stratified split
- Models: VADER (baseline, no training), BERT (bert-base-uncased), RoBERTa (roberta-base), DistilBERT (distilbert-base-uncased)
- Fine-tuning: 3 epochs, learning rate 2e-5, warmup ratio 0.1, weight decay 0.01, batch size 16, max sequence length 256
- Evaluation: Accuracy, Macro Precision, Macro Recall, Macro F1-score
- Framework: HuggingFace Transformers, PyTorch, Google Colab (T4 GPU)

**Results:**
- Report the table from final_results.csv
- Include model_comparison_chart.png and f1_ranking.png as figures
- Include one confusion matrix figure per model
- State which model performed best and by how much

**Conclusion:**
- State the winning model and its F1-score
- Compare to the VADER baseline to show improvement from fine-tuning
- Link back to your research questions
- Limitations: VADER auto-labelling introduces noise; single dataset; English only
- Future work: multilingual models, explainability (SHAP/LIME), real-time monitoring